# Session 2 — BERT Named Entity Recognition (NER)
**Task:** Label each token as a named entity (Person, Organization, Location, etc.)  
**Model:** `bert-base-cased` → `BertForTokenClassification`  
**Dataset:** CoNLL-2003  
**Metric:** F1 (seqeval)

---
### Key difference from Classification
- Session 1: one label per **sentence** (sentence-level)  
- Session 2: one label per **token** (token-level) — BERT outputs a vector per token, each gets its own classification head

### IOB tagging scheme
- `O` — not an entity  
- `B-PER` — beginning of a Person entity  
- `I-PER` — inside a Person entity  
- `B-ORG`, `I-ORG` — Organization  
- `B-LOC`, `I-LOC` — Location  
- `B-MISC`, `I-MISC` — Miscellaneous

## Step 1 — Imports & Config

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset

DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "bert-base-cased"   # cased — NER is case-sensitive (London ≠ london)
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 3
LR         = 2e-5
TRAIN_SIZE = 3000
VAL_SIZE   = 500
SAVE_DIR   = "../../models/05_transformers/bert_ner"

print(f"Device: {DEVICE}")

## Step 2 — Load & Inspect Dataset
CoNLL-2003: each example is a list of words with corresponding NER tags.

In [ ]:
raw      = load_dataset("conll2003", trust_remote_code=True)
label_names = raw["train"].features["ner_tags"].feature.names
num_labels  = len(label_names)

print("Labels:", label_names)
print("Num labels:", num_labels)

# Inspect one example
ex = raw["train"][0]
print("\nTokens:", ex["tokens"])
print("NER tags (ids):", ex["ner_tags"])
print("NER tags (names):", [label_names[t] for t in ex["ner_tags"]])

## Step 3 — Tokenizer & Label Alignment
**The tricky part of NER:** BERT uses WordPiece — one word can split into multiple subword tokens.  
e.g. `"Washington"` → `["Washington"]` but `"playing"` → `["play", "##ing"]`

We only label the **first subword** of each word. All continuation subwords (`##...`) get label `-100` so PyTorch ignores them in the loss.

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# Show the alignment problem
words  = ["London", "is", "playing", "host"]
labels = [3, 0, 0, 0]   # B-LOC, O, O, O

enc = tokenizer(words, is_split_into_words=True, return_offsets_mapping=True)
print("Tokens:      ", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("word_ids:    ", enc.word_ids())   # which original word each token belongs to

# Align labels: first subword gets the label, rest get -100
aligned = []
prev_word_id = None
for word_id in enc.word_ids():
    if word_id is None:
        aligned.append(-100)   # [CLS] and [SEP]
    elif word_id != prev_word_id:
        aligned.append(labels[word_id])
    else:
        aligned.append(-100)   # continuation subword — ignored in loss
    prev_word_id = word_id

print("Aligned labels:", aligned)

## Step 4 — Dataset

In [ ]:
class NERDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_len=MAX_LEN):
        self.data      = hf_split
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item   = self.data[idx]
        words  = item["tokens"]
        labels = item["ner_tags"]

        enc = self.tokenizer(
            words,
            is_split_into_words=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        # Align labels
        aligned, prev_word_id = [], None
        for word_id in enc.word_ids():
            if word_id is None:
                aligned.append(-100)
            elif word_id != prev_word_id:
                aligned.append(labels[word_id])
            else:
                aligned.append(-100)
            prev_word_id = word_id

        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(aligned, dtype=torch.long),
        }

train_raw = raw["train"].select(range(TRAIN_SIZE))
val_raw   = raw["validation"].select(range(VAL_SIZE))

train_ds = NERDataset(train_raw, tokenizer)
val_ds   = NERDataset(val_raw,   tokenizer)

item = train_ds[0]
print("input_ids shape:", item["input_ids"].shape)
print("labels shape:   ", item["labels"].shape)
print("labels sample:  ", item["labels"][:15].tolist())

## Step 5 — Model, Optimizer, Scheduler

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = BertForTokenClassification.from_pretrained(MODEL_NAME, num_labels=num_labels).to(DEVICE)

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print(f"Model: BertForTokenClassification  |  Labels: {num_labels}  |  Device: {DEVICE}")

## Step 6 — Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds  = logits.argmax(dim=-1)

            # Flatten — skip -100 (subwords / special tokens)
            for pred_seq, label_seq in zip(preds, labels):
                for p, l in zip(pred_seq, label_seq):
                    if l.item() != -100:
                        all_preds.append(label_names[p.item()])
                        all_labels.append(label_names[l.item()])

    # Token-level accuracy (simple metric — use seqeval for proper F1)
    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    return correct / len(all_labels)


for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, scheduler)
    acc  = evaluate(model, val_loader)
    print(f"Epoch {epoch}/{EPOCHS} | loss: {loss:.4f} | token_acc: {acc:.4f}")

## Step 7 — Save & Inference

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")


def predict_ner(text, model, tokenizer):
    model.eval()
    words = text.split()
    enc   = tokenizer(words, is_split_into_words=True, return_tensors="pt",
                      truncation=True, max_length=MAX_LEN)
    word_ids = enc.word_ids()

    input_ids      = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    preds = logits.argmax(dim=-1)[0].tolist()

    # Map back to words (first subword only)
    results, seen = [], set()
    for token_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in seen:
            results.append((words[word_id], label_names[preds[token_idx]]))
            seen.add(word_id)
    return results


sentences = [
    "Elon Musk founded SpaceX in Los Angeles.",
    "Barack Obama was born in Hawaii and studied at Harvard.",
]
for sent in sentences:
    print(f"\n{sent}")
    for word, tag in predict_ner(sent, model, tokenizer):
        if tag != "O":
            print(f"  {word:15s} → {tag}")